#### Стеганография аудио и видео информации. Лабораторная работа №6
|   Группа          |   ФИО             |   
|   :------------:  |   :------------:  |
|   М092501(71)     |   Шарибжанов И.Т. |

**Цель работы:**
Изучение метода скрытия данных в частотной области методом Коха и Жао, получение навыков реализации изученного метода в программной среде

##### Импорт необходимых библиотек

In [16]:
import cv2
import numpy as np
from scipy.fftpack import dct, idct
import os

##### Вспомогательные функции для ДКП и работы с текстом

In [17]:
def dct2(block):
    """2D Дискретное косинусное преобразование"""
    return dct(dct(block.T, norm='ortho').T, norm='ortho')

def idct2(block):
    """Обратное 2D ДКП"""
    return idct(idct(block.T, norm='ortho').T, norm='ortho')

def text_to_bits(text):
    """Преобразование текста в биты с использованием UTF-8 кодировки"""
    bytes_data = text.encode('utf-8')
    return ''.join([f'{b:08b}' for b in bytes_data])

def bits_to_text(bits):
    """Преобразование бит обратно в UTF-8 текст"""
    # Отбрасываем неполные байты в конце (если есть)
    bits = bits[:len(bits) - (len(bits) % 8)]
    bytes_data = bytearray()
    for i in range(0, len(bits), 8):
        bytes_data.append(int(bits[i:i+8], 2))
    return bytes_data.decode('utf-8', 'ignore')

##### Реализация алгоритма встраивания (Метод Коха и Жао)

In [18]:
def embed_koch_zhao(image_path, secret_text, output_path, P=40):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Файл {image_path} не найден")
    
    blue_channel = np.float32(img[:, :, 0])
    h, w = blue_channel.shape
    
    binary_message = text_to_bits(secret_text + "###EOF") 
    
    u1, v1 = 4, 5
    u2, v2 = 5, 4
    
    bit_idx = 0
    margin = P + 5 # Запас для компенсации погрешностей округления uint8
    
    for i in range(0, h - 7, 8):
        for j in range(0, w - 7, 8):
            if bit_idx < len(binary_message):
                block = blue_channel[i:i+8, j:j+8]
                dct_block = dct2(block)
                
                bit = binary_message[bit_idx]
                c1 = dct_block[u1, v1]
                c2 = dct_block[u2, v2]
                abs_c1 = abs(c1)
                abs_c2 = abs(c2)
                
                # Встраивание согласно условиям разницы коэффициентов
                if bit == '0':
                    if abs_c1 - abs_c2 < margin:
                        avg = (abs_c1 + abs_c2) / 2.0
                        abs_c1 = avg + margin / 2.0
                        abs_c2 = avg - margin / 2.0
                        if abs_c2 < 0:
                            abs_c1 += abs(abs_c2)
                            abs_c2 = 0
                else: # bit == '1'
                    if abs_c2 - abs_c1 < margin:
                        avg = (abs_c1 + abs_c2) / 2.0
                        abs_c2 = avg + margin / 2.0
                        abs_c1 = avg - margin / 2.0
                        if abs_c1 < 0:
                            abs_c2 += abs(abs_c1)
                            abs_c1 = 0
                
                # Возврат знаков исходным коэффициентам
                dct_block[u1, v1] = abs_c1 if c1 >= 0 else -abs_c1
                dct_block[u2, v2] = abs_c2 if c2 >= 0 else -abs_c2
                
                blue_channel[i:i+8, j:j+8] = idct2(dct_block)
                bit_idx += 1
            else:
                break
                
    blue_channel = np.clip(blue_channel, 0, 255)
    img[:, :, 0] = np.uint8(blue_channel)
    cv2.imwrite(output_path, img)
    print(f"Данные скрыты в {output_path}. Использовано бит: {bit_idx}")

##### Реализация алгоритма извлечения

In [19]:
def extract_koch_zhao(stego_path, u1=4, v1=5, u2=5, v2=4):
    img = cv2.imread(stego_path)
    blue_channel = np.float32(img[:, :, 0])
    h, w = blue_channel.shape
    
    bits = ""
    for i in range(0, h - 7, 8):
        for j in range(0, w - 7, 8):
            block = blue_channel[i:i+8, j:j+8]
            dct_block = dct2(block)
            
            # Значение переданного бита определяется исходя из условия
            if abs(dct_block[u1, v1]) > abs(dct_block[u2, v2]):
                bits += '0'
            else:
                bits += '1'
                
    full_text = bits_to_text(bits)
    if "###EOF" in full_text:
        return full_text.split("###EOF")[0]
    return full_text

##### Запуск и выполнение программы

In [20]:
# 1. Создание тестового файла A.bmp (если его нет)
if not os.path.exists("A.bmp"):
    # Создаем пустое синее изображение 400x400
    empty_img = np.full((400, 400, 3), (200, 100, 100), dtype=np.uint8)
    cv2.imwrite("A.bmp", empty_img)

# 2. Текст для скрытия
secret_text = "Шарибжанов"

# 3. Выполнение скрытия
embed_koch_zhao("A.bmp", secret_text, "B.bmp", P=30)

# 4. Выполнение извлечения и сохранение в A.txt 
extracted_text = extract_koch_zhao("B.bmp")
with open("A.txt", "w", encoding="utf-8") as f:
    f.write(extracted_text)

print(f"Результат извлечения: {extracted_text}")
print("Работа программы завершена")

Данные скрыты в B.bmp. Использовано бит: 208
Результат извлечения: Шарибжанов
Работа программы завершена
